### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="amazon_employee_access",
    dataset_year="2010",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/c/amazon-employee-access-challenge",
    download_description="""
mkdir -p local-data-warehouse/amazon_employee_access/ \
&& kaggle competitions download -c amazon-employee-access-challenge -f train.csv -p local-data-warehouse/amazon_employee_access/ \
&& unzip local-data-warehouse/amazon_employee_access/train.csv.zip -d local-data-warehouse/amazon_employee_access/ \
&& rm local-data-warehouse/amazon_employee_access/train.csv.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{hamner2013amazon,
  author       = {Ben Hamner and kenmonta and Will Cukierski},
  title        = {Amazon.com - Employee Access Challenge},
  year         = {2013},
  howpublished = {\url{https://www.kaggle.com/competitions/amazon-employee-access-challenge}},
  note         = {Kaggle competition},
}
""",
    academic_reference_bibtex_key="hamner2013amazon",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We only use the training data from Kaggle.
- We renamed the target "ACTION" to "ResourceApproved" and mapped binary values to "Yes"/"No".
- Anomaly: the data might contain sub-groups related to managers and resources.
- Anomaly: likely, similar to the test data, each sample represents a unique employee.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="ResourceApproved",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="ResourceApproved",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/train.csv")

target_feature = "ResourceApproved"
df = df.rename(columns={"ACTION": target_feature})
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"})

cat_features = [
    "RESOURCE",
    "MGR_ID",
    "ROLE_ROLLUP_1",
    "ROLE_ROLLUP_2",
    "ROLE_DEPTNAME",
    "ROLE_TITLE",
    "ROLE_FAMILY_DESC",
    "ROLE_FAMILY",
    "ROLE_CODE",
    "ResourceApproved",
]

df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 32,769
Columns: 10
Use sampling: False (sample size: 32,769)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['RESOURCE', 'MGR_ID', 'ROLE_FAMILY_DESC', 'ROLE_DEPTNAME', 'ROLE_CODE', 'ROLE_TITLE', 'ROLE_ROLLUP_2', 'ROLE_ROLLUP_1', 'ROLE_FAMILY']
Rows remaining as candidates after top-9 filter: 0 (of 32,769)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,ResourceApproved,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
0,Yes,37793,81744,117902,117903,118783,118451,130134,118453,118454
1,Yes,40309,1541,117961,118225,123173,119093,123174,119095,119096
2,Yes,27356,205,117961,118386,118746,118784,147114,290919,118786
3,Yes,5173,8229,117961,118300,121305,119351,149246,3130,119353
4,Yes,77207,51791,117961,119256,120943,118995,280788,292795,118997


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,ResourceApproved,category,0.0,0.0,2.0,"Yes, No"
1,RESOURCE,category,0.0,0.0,7518.0,"4675, 79092, 75078, 25993, 3853, 75834, 6977, 32270, 42085, 17308"
2,MGR_ID,category,0.0,0.0,4243.0,"770, 2270, 2594, 1350, 2014, 16850, 3966, 7807, 5244, 5396"
3,ROLE_ROLLUP_1,category,0.0,0.0,128.0,"117961, 117902, 91261, 118315, 118212, 118290, 119062, 118887, 117916, 118169"
4,ROLE_ROLLUP_2,category,0.0,0.0,177.0,"118300, 118343, 118327, 118225, 118386, 118052, 117962, 118413, 118446, 118026"
5,ROLE_DEPTNAME,category,0.0,0.0,449.0,"117878, 117941, 117945, 118514, 117920, 117884, 119598, 118403, 119181, 120722"
6,ROLE_TITLE,category,0.0,0.0,343.0,"118321, 117905, 118784, 117879, 118568, 117885, 118054, 118685, 118777, 118451"
7,ROLE_FAMILY_DESC,category,0.0,0.0,2358.0,"117906, 240983, 117913, 279443, 117886, 130134, 117897, 117879, 168365, 133686"
8,ROLE_FAMILY,category,0.0,0.0,67.0,"290919, 118424, 19721, 117887, 292795, 118398, 308574, 118453, 118331, 118643"
9,ROLE_CODE,category,0.0,0.0,343.0,"118322, 117908, 118786, 117880, 118570, 117888, 118055, 118687, 118779, 118454"


In [6]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column           rank                      
MGR_ID           1        770    152   0.46
                 2       2270     99   0.30
                 3       2594     82   0.25
                 4       1350     71   0.22
                 5       2014     67   0.20
RESOURCE         1       4675    839   2.56
                 2      79092    484   1.48
                 3      75078    409   1.25
                 4      25993    409   1.25
                 5       3853    404   1.23
ROLE_CODE        1     118322   4649  14.19
                 2     117908   3583  10.93
                 3     118786   1772   5.41
                 4     117880   1256   3.83
                 5     118570   1043   3.18
ROLE_DEPTNAME    1     117878   1135   3.46
                 2     117941    763   2.33
                 3     117945    659   2.01
                 4     118514    601   1.83
                 5     117920    597   1.82
ROLE_FAMILY      1     290919  10980  33.51
                 2     118424   2690   8.21
                 3      19721   2636   8.04
                 4     117887   2400   7.32
                 5     292795   1318   4.02
ROLE_FAMILY_DESC 1     117906   6896  21.04
                 2     240983   1244   3.80
                 3     117913    670   2.04
                 4     279443    665   2.03
                 5     117886    530   1.62
ROLE_ROLLUP_1    1     117961  21407  65.33
                 2     117902    742   2.26
                 3      91261    721   2.20
                 4     118315    498   1.52
                 5     118212    400   1.22
ROLE_ROLLUP_2    1     118300   4424  13.50
                 2     118343   3945  12.04
                 3     118327   2641   8.06
                 4     118225   2547   7.77
                 5     118386   1796   5.48
ROLE_TITLE       1     118321   4649  14.19
                 2     117905   3583  10.93
                 3     118784   1772   5.41
                 4     117879   1256   3.83
                 5     118568   1043   3.18
ResourceApproved 1        Yes  30872  94.21
                 2         No   1897   5.79

In [8]:
# Target Distribution
target_df

,count,pct
ResourceApproved,,
Yes,30872,94.21
No,1897,5.79


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to amazon_employee_access/019d7366-58b8-7e97-bf34-83c96a915561


019d7366-58b8-7e97-bf34-83c96a915561
727bc7d97d3fb42a39bbe5c3dc57c8c9dfe2763b41b895054a8d1629f2aa4207
